# Тема 7. RAG-системи

Демонстраційний код для живого заняття.


### Встановлення залежностей

In [1]:
!pip install -qU --no-warn-conflicts "langchain>=0.3.0,<1.0.0" "langchain-core>=0.3.0,<1.0.0" "langchain-community>=0.3.0,<1.0.0"
!pip install -qU --no-warn-conflicts "langchain-openai>=0.2.0" "langchain-chroma>=0.1.2" "langchain-text-splitters>=0.3.0" "langsmith>=0.1.99"
!pip install -qU --no-warn-conflicts chromadb beautifulsoup4 tiktoken requests pydantic

import warnings
warnings.filterwarnings("ignore")


  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
Using cached opentelemetry_api-1.42.1-py3-none-any.whl (61 kB)
Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl (170 kB)
Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl (203 kB)
Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
Using cached langchain_text_splitters-0.3.11-py3-none-any.whl (33 kB)
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.45.1
    Uninstalling opentelemetry-api-1.45.1:
      Successfully uninstalled opentelemetry-api-1.45.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Fou

### Налаштування змінних середовища

---



In [2]:
import os
from getpass import getpass
def get_secret(name: str, required: bool = True) -> str:
    # 1) змінна середовища, 2) Colab Secrets, 3) ручне введення
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    hint = "" if required else " (Enter — пропустити)"
    return getpass(f"Введіть {name}{hint}: ")

# Встановлення OpenRouter API ключа
os.environ['OPENROUTER_API_KEY'] = get_secret("OPENROUTER_API_KEY")
BASE_URL = "https://openrouter.ai/api/v1"
#MODEL = "openai/gpt-4o-mini"   # модель має підтримувати tool calling

# https://smith.langchain.com/
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = get_secret("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "neoversity-topic-7"

# Встановлення User-Agent для уникнення блокування при scraping веб-сторінок
os.environ['USER_AGENT'] = "myagent"

Завантаження документів


In [3]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Ініціалізація завантажувача веб-документів з конкретними інструкціями парсингу
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

# Завантаження відфільтрованого контенту з веб-сторінки в документи
docs = loader.load()

Розбиття тексту на частини


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Створення розділювача тексту для поділу тексту на фрагменти по 1000 символів з перекриттям 200 символів
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Розділення завантажених документів на менші фрагменти
splits = text_splitter.split_documents(docs)

Робота з векторною базою


In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",  # Модель ембедингів, що підтримується на OpenRouter
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ.get("OPENROUTER_API_KEY")  # Ваш ключ OpenRouter, записаний у змінні середовища
)

# Ембедінг текстових фрагментів та їх збереження у векторному сховищі Chroma для пошуку за подібністю
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="topic7-rag-systems"
)


Налаштування ретрівера


In [6]:
# Створення ретрівера з векторного сховища
retriever = vectorstore.as_retriever()

Налаштування ретрівера


In [7]:
# Отримання релевантних документів для запиту
query = "What is Task Decomposition?"
retrieved_docs = retriever.invoke(query)

print(f"Знайдено документів (retriever.invoke): {len(retrieved_docs)}")
print("\nПерший документ (фрагмент):")
print(retrieved_docs[0].page_content[:600])

# Діагностика зі скорами релевантності (доступно через vectorstore)
scored_docs = vectorstore.similarity_search_with_relevance_scores(query, k=4)
print("\nТоп-4 документи зі скорами релевантності:")
for i, (doc, score) in enumerate(scored_docs, start=1):
    print(f"{i}. score={score:.4f} | source={doc.metadata.get('source', 'n/a')}")
    print(f"   preview: {doc.page_content[:180].replace('\n', ' ')}")

# Проста евристика coverage: чи є в контексті терміни, потрібні для відповіді
expected_terms = ["task decomposition", "tree of thoughts", "chain of thought"]
retrieved_text = "\n".join(doc.page_content.lower() for doc in retrieved_docs)
coverage = {term: (term in retrieved_text) for term in expected_terms}
print("\nCoverage очікуваних термінів:")
for term, present in coverage.items():
    print(f"- {term}: {'YES' if present else 'NO'}")


Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable planner which is common in certain robotic setups but not in many other domains.
Self-Reflection#


Завантаження промпту з Hub


In [8]:
from langsmith import Client

# Ініціалізуємо офіційний клієнт LangSmith
client = Client()

# Завантажуємо публічний промпт напряму через метод клієнта
prompt = client.pull_prompt(
    "rlm/rag-prompt", dangerously_pull_public_prompt=True
)

# Форматований вивід інформації про промпт
print("=== МЕТАДАНІ ПРОМПТУ ===")
print(f"Вхідні змінні: {prompt.input_variables}")
if hasattr(prompt, 'metadata') and prompt.metadata:
    print(f"Репозиторій: {prompt.metadata.get('lc_hub_repo')}")
    print(f"Автор: {prompt.metadata.get('lc_hub_owner')}")

print("\n=== ШАБЛОН ПОВІДОМЛЕННЯ (TEMPLATE) ===")
# Проходимо по повідомленнях і виводимо сам шаблон
for i, message in enumerate(prompt.messages, start=1):
    print(f"Повідомлення #{i} ({type(message).__name__}):")
    print("-" * 40)
    if hasattr(message, 'prompt') and hasattr(message.prompt, 'template'):
        print(message.prompt.template)
    else:
        print(message)
    print("-" * 40)

=== МЕТАДАНІ ПРОМПТУ ===
Вхідні змінні: ['context', 'question']
Репозиторій: rag-prompt
Автор: rlm

=== ШАБЛОН ПОВІДОМЛЕННЯ (TEMPLATE) ===
Повідомлення #1 (HumanMessagePromptTemplate):
----------------------------------------
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
----------------------------------------


Ініціалізація моделі


In [9]:
from langchain_openai import ChatOpenAI

# Ініціалізація LLM
llm = ChatOpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY") , # ключ OpenRouter
    base_url="https://openrouter.ai/api/v1",       # завжди OpenRouter
    model="openai/gpt-4o-mini",                    # модель через OpenRouter
    temperature=0
)


Налаштування ретрівера


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Допоміжна функція для форматування отриманих документів
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Базовий RAG-ланцюг (повертає строку)
rag_chain_basic = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}  # Передає запит користувача далі без жодних модифікацій
    | prompt
    | llm
    | StrOutputParser()
)

# Базовий RAG-ланцюг (повертає AIMessage)
rag_chain_basic_message = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)


### Тема 7. RAG

1. Запитати чи були проблеми з запуском коду з конспекту. Чи все зрозуміло? При потребі ще раз пояснити базову імплементацію RAG перед тим як показувати складніші.

### Просунуті трансформації запитів

#### Multi-Query Generation

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# Промпт для генерації множинних запитів
template = """You are an AI language assistant model.
 Your task is to generate five different versions of a given user
 question to retrieve relevant documents from a vector database.
 By generating multiple perspectives of the user's question,
 your goal is to help the user overcome some of the limitations of distance-based similarity search.
 Provide these alternative questions, separated by newlines. Original Question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

# Ланцюг для генерації запитів
generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.split("\n") if q.strip()]) # Фільтруємо порожні рядки
)

Виклик моделі


In [12]:
question = "What are the main components of an LLM-powered autonomous agent system?" # What is Decomposition? - alternative question to test
generated_queries_list = generate_queries.invoke({"question": question})

# Виведення згенерованих запитів
for i, q in enumerate(generated_queries_list):
    print(f"{i+1}. {q}")

#### ВИВІД ####
# 1. How does task decomposition work in the context of LLM agents?
# 2. Can you explain the concept of task decomposition as it relates to LLM agents?
# 3. What are the principles of task decomposition when applied to LLM agents?
# 4. In what ways does task decomposition benefit LLM agents?
# 5. What role does task decomposition play in the functionality of LLM agents?

1. 1. What key elements make up a system for autonomous agents powered by large language models (LLMs)?
2. 2. Can you outline the primary features of an autonomous agent system that utilizes LLM technology?
3. 3. What are the essential building blocks of a system that enables autonomous agents to function using large language models?
4. 4. Which core components are involved in the design of LLM-driven autonomous agent systems?
5. 5. What are the fundamental parts of a system that supports autonomous agents with large language model capabilities?


Налаштування ретрівера


In [13]:
from langchain_core.load import dumps, loads
from langchain_core.documents import Document

def get_unique_union(documents: list[list]):
    """Проста функція для отримання унікального об'єднання отриманих документів"""
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))
    return [loads(doc, allowed_objects=[Document]) for doc in unique_docs]

# Побудова ланцюга отримання (Multi-Query)
retrieval_chain_multi_query = generate_queries | retriever.map() | get_unique_union

# Виклик ланцюга та перевірка кількості отриманих документів
docs = retrieval_chain_multi_query.invoke({"question": question})
print(f"Всього унікальних документів отримано: {len(docs)}")


Всього унікальних документів отримано: 6


/tmp/ipykernel_23240/3442729060.py:8: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc, allowed_objects=[Document]) for doc in unique_docs]


#### RAG-Fusion

In [14]:
def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal Rank Fusion that intelligently combines multiple ranked lists """
    fused_scores = {}

    # Проходимо по списку документів, відсортованих за рейтингом
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Суть алгоритму RRF: документи з вищим рейтингом (меншим значенням) отримують більшу кількість балів
            fused_scores[doc_str] += 1 / (rank + k)

    # Сортування документів за їхніми новими об’єднаними балами у порядку спадання
    reranked_results = [
        (loads(doc_str, allowed_objects=[Document]), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

Створення та виконання промпту


In [15]:
# Використовуємо дещо інший запит для RAG-Fusion
template = """You are a helpful assistant that generates multiple search queries based on a single input query. 

Generate multiple search queries related to: {question} 

Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

generate_queries_rag_fusion = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

# Новий ланцюжок пошуку за допомогою RRF
retrieval_chain_rag_fusion = generate_queries_rag_fusion | retriever.map() | reciprocal_rank_fusion
rag_fusion_ranked_docs = retrieval_chain_rag_fusion.invoke({"question": question})

print(f"Total re-ranked documents retrieved: {len(rag_fusion_ranked_docs)}")


#### ВИВІД ####
# Total re-ranked documents retrieved: 7


Total re-ranked documents retrieved: 6


In [16]:
# 1. Дістаємо самі об'єкти Document з результатів RAG-Fusion (оскільки там пари (Document, score))
fusion_documents = [doc for doc, score in rag_fusion_ranked_docs]

# 2. Форматуємо їх у єдиний рядок контексту
formatted_fusion_context = format_docs(fusion_documents)

# 3. Використовуємо окремий ланцюжок повідомлень без StrOutputParser для отримання об'єкта AIMessage
rag_fusion_answer_chain = (
    prompt
    | llm
)

# 4. Викликаємо ланцюжок із отриманим контекстом та оригінальним питанням
result_message = rag_fusion_answer_chain.invoke({"context": formatted_fusion_context, "question": question})

# 5. Красиве виведення результату
messages = [result_message]
for message in messages:
    message.pretty_print()


================================== Ai Message ==================================

The main components of an LLM-powered autonomous agent system include memory and tool use. Memory consists of short-term memory for in-context learning and long-term memory for retaining and recalling information over extended periods. Tool use enables the agent to call external APIs for additional information and capabilities beyond its pre-trained model.


#### Decomposition

In [17]:
# Decomposition prompt
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

# Ланцюжок для формування підпитань
generate_queries_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

# Створити та роздрукувати підпитання
question = "What are the main components of an LLM-powered autonomous agent system?"
sub_questions = generate_queries_decomposition.invoke({"question": question})
print("\n" + "\n".join(sub_questions))


#### ВИВІД ####
# [
#  '1. What are the key components ... ?',
#  '2. How do perception and decision-making modules ... ?',
#  '3. What role does reinforcement learning play in enhancing ... ?'
# ]


1. What are the key components of a large language model (LLM) used in autonomous agent systems?
2. How do perception and sensor integration function in LLM-powered autonomous agents?
3. What role does decision-making play in the architecture of LLM-based autonomous agents?
4. How is natural language processing implemented in LLM-powered autonomous agent systems?


Створення та виконання промпту


In [18]:
# RAG prompt
prompt_rag = client.pull_prompt(
    "rlm/rag-prompt", dangerously_pull_public_prompt=True
)

# Список, у якому містяться відповіді на наші підпитання
rag_results = []
for sub_question in sub_questions:
    # Отримання документів для кожного підпитання
    retrieved_docs = retriever.invoke(sub_question)

    # Ланцюжок RAG, щоб відповісти на це підпитання
    answer = (prompt_rag | llm | StrOutputParser()).invoke(
        {"context": format_docs(retrieved_docs), "question": sub_question}
    )
    rag_results.append(answer)

def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

# Об’єднання пари «питання-відповідь» в один контекстний рядок
context = format_qa_pairs(sub_questions, rag_results)

# Фінальний промпт для синтезу
decomposition_synthesis_template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the original question: {question}
"""
decomposition_synthesis_prompt = ChatPromptTemplate.from_template(decomposition_synthesis_template)

# Окремий ланцюжок для Decomposition
decomposition_answer_chain = (
    decomposition_synthesis_prompt
    | llm
)

# Викликаємо ланцюжок та зберігаємо об'єкт AIMessage
result_message = decomposition_answer_chain.invoke({"context": context, "question": question})

# Красиве виведення за допомогою pretty_print
messages = [result_message]
for message in messages:
    message.pretty_print()


================================== Ai Message ==================================

The main components of an LLM-powered autonomous agent system include:

1. **Planning**: This involves breaking down tasks into manageable subgoals and refining actions through self-reflection, which enhances the agent's ability to solve complex problems effectively.

2. **Memory**: The system incorporates both short-term and long-term memory capabilities, allowing the agent to learn, retain information, and recall relevant data to improve decision-making and performance.

3. **Tool Use**: LLM-powered agents can access external APIs and tools to supplement their knowledge and enhance their capabilities in real-time tasks.

4. **Perception and Sensor Integration**: The LLM acts as the central controller, processing sensory inputs and coordinating actions based on the information retained in memory, which improves the agent's decision-making abilities.

5. **Natural Language Processing (NLP)**: NLP serves a

#### Step-Back Prompting

In [19]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate


# Few-shot приклади для навчання моделі генеруванню запитань, що вимагають переосмислення (більш загальних)
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel's was born in what country?",
        "output": "what is Jan Sindel's personal history?",
    },
]

# Визначення як має бути оформлено кожен приклад у запиті
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),  # User input
    ("ai", "{output}")     # Model's response
])

# Обгортання few-shot прикладів у шаблон промпта
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Повний промпт, який містить системні інструкції, few-shot приклади, і запитання користувача
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert at world knowledge. Your task is to step back and paraphrase a question "
     "to a more generic step-back question, which is easier to answer. Here are a few examples:"),
    few_shot_prompt,
    ("user", "{question}"),
])

Виконання ланцюжка


In [20]:
# Визначення ланцюжка
generate_queries_step_back = prompt | llm | StrOutputParser()

# Запуск з запитанням
question = "What are the main components of an LLM-powered autonomous agent system?"
step_back_question = generate_queries_step_back.invoke({"question": question})

# Вивід оригінального і згенерованого запитань
print(f"Original Question: {question}")
print(f"Step-Back Question: {step_back_question}")


#### ВИВІД ####
# Original Question: What is task decomposition for LLM agents?
# Step-Back Question:  what does task decomposition mean in the context of agents?

Original Question: What are the main components of an LLM-powered autonomous agent system?
Step-Back Question: what are the key elements of an autonomous agent system?


Створення та виконання промпту


In [21]:
from langchain_core.runnables import RunnableLambda


# Промпт для фінальної відповіді
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question.
Your response should be comprehensive and not contradicted with the following context if they are relevant.
Otherwise, ignore them if they are not relevant.

# Normal Context
{normal_context}

# Step-Back Context
{step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

# Окремий ланцюжок Step-Back
step_back_answer_chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the original question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm

)

result_message = step_back_answer_chain.invoke({"question": question})

# Оскільки результат є поодиноким об'єктом AIMessage, ми можемо загорнути його у список та вивести через pretty_print()
messages = [result_message]

for message in messages:
    message.pretty_print()


#### ВИВІД ####
# Task decomposition for LLM (large language model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This is


================================== Ai Message ==================================

An LLM-powered autonomous agent system consists of several key components that work together to enable the agent to function effectively. These components include:

1. **Planning**:
   - **Subgoal and Decomposition**: The agent is capable of breaking down large tasks into smaller, manageable subgoals. This allows for more efficient handling of complex tasks by addressing them step-by-step.
   - **Reflection and Refinement**: The agent can engage in self-criticism and self-reflection regarding its past actions. This capability enables the agent to learn from its mistakes and refine its approach for future tasks, ultimately improving the quality of its results.

2. **Memory**:
   - **Short-term Memory**: This involves in-context learning, where the model utilizes its immediate context to learn and adapt during interactions.
   - **Long-term Memory**: The agent can retain and recall information over extended

Виконання коду


#### HyDE

In [22]:
# HyDE prompt
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

# Ланцюжок для генерації гіпотетичного документа
generate_docs_for_retrieval = (
    prompt_hyde
    | llm
    | StrOutputParser()
)

# Генерація і вивід гіпотетичного документу
hypothetical_document = generate_docs_for_retrieval.invoke({"question": question})
print(hypothetical_document)


#### ВИВІД ####
# **Task Decomposition for LLM .... **

# Task decomposition is a critical  ....

**Title: Components of an LLM-Powered Autonomous Agent System**

**Abstract:** The advent of Large Language Models (LLMs) has revolutionized the development of autonomous agent systems, enabling them to perform complex tasks with a high degree of sophistication. This paper delineates the primary components that constitute an LLM-powered autonomous agent system, highlighting their interdependencies and functionalities.

**1. Introduction**  
Autonomous agents are systems capable of performing tasks without human intervention, leveraging artificial intelligence to make decisions based on environmental inputs. The integration of LLMs into these systems enhances their ability to understand and generate human-like text, facilitating more natural interactions and improved decision-making processes.

**2. Main Components of an LLM-Powered Autonomous Agent System**

**2.1. Large Language Model (LLM)**  
At the core of the system is the LLM, which serves as the primary engine for natural langua

Налаштування ретрівера


In [23]:
# Отримання документів за допомогою підходу HyDE
retrieval_chain_hyde = generate_docs_for_retrieval | retriever
retrieved_docs_hyde = retrieval_chain_hyde.invoke({"question": question})

print(f"HyDE retrieved docs: {len(retrieved_docs_hyde)}")
for i, doc in enumerate(retrieved_docs_hyde[:3], start=1):
    print(f"{i}. {doc.page_content[:180].replace('\n', ' ')}")

# Окремий ланцюжок для фінальної відповіді HyDE (щоб уникнути конфлікту з іншими rag_chain)
hyde_answer_chain = prompt | llm

result_message = hyde_answer_chain.invoke({
    "context": format_docs(retrieved_docs_hyde),
    "question": question
})

messages = [result_message]
for message in messages:
    message.pretty_print()


================================== Ai Message ==================================

The main components of an LLM-powered autonomous agent system include:

1. **Planning**: This involves breaking down large tasks into smaller, manageable subgoals, which allows the agent to handle complex tasks more efficiently.

2. **Subgoal and Decomposition**: The agent can decompose tasks into subgoals, facilitating a structured approach to problem-solving.

3. **Reflection and Refinement**: The agent is capable of self-criticism and reflection on past actions, enabling it to learn from mistakes and improve future performance.

4. **Memory**: The system may incorporate memory components to retain information and context, enhancing the agent's ability to perform tasks over time.

5. **Natural Language Interface**: The agent relies on natural language as an interface to communicate with external components, although this can introduce reliability issues due to potential formatting errors and unpredictab

### Оцінювання RAG (Evaluation)

Додаємо базовий evaluation-контур для retrieval і generation.

- Retrieval: `Recall@K`, `MRR`
- Generation: `faithfulness` (відповідь спирається на контекст), `answer relevance` (відповідь відповідає питанню)

> Це шаблон для систематичної перевірки. Для продакшену підготуйте репрезентативний набір питань і золоті анотації.


In [ ]:
import numpy as np
from pydantic import BaseModel, Field

# Невеликий тестовий набір: питання + фрагмент, який очікуємо знайти в retrieval
# (підставте власні "gold"-мітки для вашого домену)
eval_dataset = [
    {
        "question": "What is Task Decomposition?",
        "gold_snippet": "task decomposition",
    },
    {
        "question": "What are the main components of an LLM-powered autonomous agent system?",
        "gold_snippet": "planning",
    },
    {
        "question": "How does memory work in LLM agents?",
        "gold_snippet": "memory",
    },
]


def first_relevant_rank(scored_docs, gold_snippet: str):
    gold = gold_snippet.lower()
    for idx, (doc, _score) in enumerate(scored_docs, start=1):
        if gold in doc.page_content.lower():
            return idx
    return None


def compute_retrieval_metrics(dataset, k=5):
    recall_hits = 0
    reciprocal_ranks = []

    for item in dataset:
        scored = vectorstore.similarity_search_with_relevance_scores(item["question"], k=k)
        rank = first_relevant_rank(scored, item["gold_snippet"])

        if rank is not None:
            recall_hits += 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    recall_at_k = recall_hits / len(dataset)
    mrr = float(np.mean(reciprocal_ranks))
    return recall_at_k, mrr


class FaithfulnessScore(BaseModel):
    score: int = Field(ge=1, le=5, description="Faithfulness score from 1 to 5")
    rationale: str


class AnswerRelevanceScore(BaseModel):
    score: int = Field(ge=1, le=5, description="Answer relevance score from 1 to 5")
    rationale: str


faithfulness_grader = llm.with_structured_output(FaithfulnessScore)
relevance_grader = llm.with_structured_output(AnswerRelevanceScore)


def evaluate_generation(dataset):
    faithfulness_scores = []
    relevance_scores = []

    for item in dataset:
        q = item["question"]
        ctx_docs = retriever.invoke(q)
        ctx = format_docs(ctx_docs)
        ans = rag_chain_basic.invoke(q)

        faithfulness_prompt = f"""You are evaluating a RAG answer.
Question: {q}
Context:
{ctx}
Answer: {ans}

Score faithfulness from 1 to 5:
- 5: fully supported by context
- 3: partially supported
- 1: unsupported / hallucinated"""

        relevance_prompt = f"""You are evaluating answer relevance.
Question: {q}
Answer: {ans}

Score relevance from 1 to 5:
- 5: directly and completely answers the question
- 3: partially answers
- 1: mostly irrelevant"""

        faithfulness = faithfulness_grader.invoke(faithfulness_prompt)
        relevance = relevance_grader.invoke(relevance_prompt)

        faithfulness_scores.append(faithfulness.score)
        relevance_scores.append(relevance.score)

        print("=" * 80)
        print(f"Q: {q}")
        print(f"A: {ans}")
        print(f"Faithfulness: {faithfulness.score}/5 | {faithfulness.rationale}")
        print(f"Answer relevance: {relevance.score}/5 | {relevance.rationale}")

    return float(np.mean(faithfulness_scores)), float(np.mean(relevance_scores))


recall_at_5, mrr_at_5 = compute_retrieval_metrics(eval_dataset, k=5)
mean_faithfulness, mean_answer_relevance = evaluate_generation(eval_dataset)

print("
" + "#" * 80)
print(f"Recall@5: {recall_at_5:.3f}")
print(f"MRR@5: {mrr_at_5:.3f}")
print(f"Mean faithfulness: {mean_faithfulness:.3f}")
print(f"Mean answer relevance: {mean_answer_relevance:.3f}")
